# MOD-08 · NB-05 — FastAPI for Clinical Model Serving
### Health Analytics with Python · Module 08: Reproducible Research & Deployment
---
**Learning objectives**
- Build a production FastAPI REST API for clinical model serving
- Implement Pydantic schemas for request/response validation
- Create single and batch prediction endpoints
- Write an API test suite with edge cases and error handling
- Add middleware: CORS, logging, rate limiting, authentication
- Containerise with Docker and docker-compose for deployment

**Estimated time:** 3 hours | **Level:** Advanced | **Libraries:** `fastapi`, `pydantic`, `uvicorn`

## 1. Setup — Train & Save Model

In [ ]:
import os, json, pickle, time
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
os.makedirs("/tmp/mod08", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white"})
np.random.seed(42); N = 2000
def sigmoid(x): return 1/(1+np.exp(-x))
age = np.random.normal(60,15,N).clip(18,90).astype(int)
los = np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
dm  = np.random.binomial(1,0.35,N)
hf  = np.random.binomial(1,0.22,N)
copd= np.random.binomial(1,0.18,N)
logit=-3.2+0.6*hf+0.5*dm+0.3*copd+0.02*(age-60)+0.2*(los>7).astype(int)
readmitted = np.random.binomial(1,sigmoid(logit),N)
df = pd.DataFrame({"age":age,"los_days":los,"diabetes":dm,"hf":hf,"copd":copd,"readmitted":readmitted})
FEATURES = ["age","los_days","diabetes","hf","copd"]
X_tr,X_te,y_tr,y_te = train_test_split(df[FEATURES],df["readmitted"],test_size=0.2,stratify=df["readmitted"],random_state=42)
model = GradientBoostingClassifier(n_estimators=200,max_depth=4,learning_rate=0.05,random_state=42)
model.fit(X_tr,y_tr)
auc = roc_auc_score(y_te,model.predict_proba(X_te)[:,1])
model_path = Path("/tmp/mod08/readmission_model.pkl")
pickle.dump(model, open(model_path,"wb"))
print(f"Model trained | AUC={auc:.4f} | Saved: {model_path}")


## 2. FastAPI Architecture for Clinical APIs

```
POST /predict           <- Single patient prediction
POST /predict/batch     <- Batch predictions (list of patients)
GET  /health            <- Health check (for Docker/k8s)
GET  /model/info        <- Model metadata
GET  /docs              <- Swagger UI (auto-generated)
GET  /metrics           <- Prometheus metrics (optional)

Request flow:
  HTTP Request
    -> Pydantic validation (raise 422 if invalid)
    -> Middleware (auth, logging, rate limit)
    -> Endpoint function
    -> Model.predict_proba()
    -> Pydantic response serialisation
    -> HTTP Response (JSON)
```


## 3. Complete FastAPI Application

In [ ]:
FASTAPI_APP = (
    "# main.py -- FastAPI Clinical Readmission API\n"
    "from fastapi import FastAPI, HTTPException, Depends, BackgroundTasks\n"
    "from fastapi.middleware.cors import CORSMiddleware\n"
    "from pydantic import BaseModel, Field, validator\n"
    "from typing import List, Optional\n"
    "import pickle, json, logging, time\n"
    "import numpy as np\n"
    "from datetime import datetime\n"
    "from pathlib import Path\n\n"
    "# ── App configuration ─────────────────────────────────────────────\n"
    "app = FastAPI(\n"
    "    title='Readmission Risk API',\n"
    "    description='Clinical model serving 30-day readmission risk scores',\n"
    "    version='1.0.0',\n"
    "    docs_url='/docs',       # Swagger UI\n"
    "    redoc_url='/redoc',     # ReDoc\n"
    ")\n\n"
    "app.add_middleware(CORSMiddleware, allow_origins=['*'],\n"
    "    allow_credentials=True, allow_methods=['*'], allow_headers=['*'])\n\n"
    "# ── Request / Response schemas ─────────────────────────────────────\n"
    "class PatientFeatures(BaseModel):\n"
    "    age: int = Field(..., ge=18, le=90, description='Patient age in years')\n"
    "    los_days: int = Field(..., ge=1, le=60, description='Length of stay in days')\n"
    "    diabetes: int = Field(..., ge=0, le=1, description='Diabetes diagnosis (0/1)')\n"
    "    hf: int = Field(..., ge=0, le=1, description='Heart failure diagnosis (0/1)')\n"
    "    copd: int = Field(..., ge=0, le=1, description='COPD diagnosis (0/1)')\n"
    "    patient_id: Optional[str] = Field(None, description='Optional patient identifier')\n\n"
    "class BatchPredictionRequest(BaseModel):\n"
    "    patients: List[PatientFeatures]\n\n"
    "class PredictionResponse(BaseModel):\n"
    "    patient_id: Optional[str]\n"
    "    readmission_probability: float\n"
    "    risk_tier: str           # Low / Moderate / High / Very High\n"
    "    model_version: str\n"
    "    prediction_timestamp: str\n\n"
    "# ── Model loader ────────────────────────────────────────────────────\n"
    "MODEL = None\n"
    "def load_model():\n"
    "    global MODEL\n"
    "    if MODEL is None:\n"
    "        MODEL = pickle.load(open('models/readmission_model.pkl','rb'))\n"
    "    return MODEL\n\n"
    "# ── Endpoints ───────────────────────────────────────────────────────\n"
    "@app.get('/health')\n"
    "def health_check():\n"
    "    return {'status':'healthy','timestamp':datetime.now().isoformat()}\n\n"
    "@app.get('/model/info')\n"
    "def model_info():\n"
    "    return {'model_type':'GradientBoostingClassifier',\n"
    "            'version':'1.0.0','auc':0.789,\n"
    "            'features':['age','los_days','diabetes','hf','copd'],\n"
    "            'trained_on':'2024-01-15','n_training':2400}\n\n"
    "@app.post('/predict', response_model=PredictionResponse)\n"
    "def predict_single(patient: PatientFeatures, model=Depends(load_model)):\n"
    "    X = [[patient.age, patient.los_days, patient.diabetes, patient.hf, patient.copd]]\n"
    "    prob = float(model.predict_proba(X)[0][1])\n"
    "    tier = ('Low' if prob<0.10 else 'Moderate' if prob<0.20\n"
    "            else 'High' if prob<0.35 else 'Very High')\n"
    "    return PredictionResponse(\n"
    "        patient_id=patient.patient_id,\n"
    "        readmission_probability=round(prob,4),\n"
    "        risk_tier=tier,\n"
    "        model_version='1.0.0',\n"
    "        prediction_timestamp=datetime.now().isoformat()\n"
    "    )\n\n"
    "@app.post('/predict/batch')\n"
    "def predict_batch(request: BatchPredictionRequest, model=Depends(load_model)):\n"
    "    results = []\n"
    "    for patient in request.patients:\n"
    "        X = [[patient.age, patient.los_days, patient.diabetes, patient.hf, patient.copd]]\n"
    "        prob = float(model.predict_proba(X)[0][1])\n"
    "        tier = ('Low' if prob<0.10 else 'Moderate' if prob<0.20\n"
    "                else 'High' if prob<0.35 else 'Very High')\n"
    "        results.append({'patient_id':patient.patient_id,\n"
    "                        'probability':round(prob,4),'risk_tier':tier})\n"
    "    return {'predictions':results,'count':len(results)}\n\n"
    "# Run: uvicorn main:app --reload --host 0.0.0.0 --port 8000\n"
)
print("FastAPI application code (main.py):")
print(FASTAPI_APP[:2000])
app_path = Path("/tmp/mod08/main.py")
app_path.write_text(FASTAPI_APP)


## 4. Pydantic Validation & Predictions

In [ ]:
try:
    from pydantic import BaseModel, Field, field_validator
    PYDANTIC_OK = True
except ImportError:
    PYDANTIC_OK = False
    print("pip install pydantic>=2.0")

# Simulate Pydantic validation without running a server
class PatientFeatures:
    def __init__(self, age, los_days, diabetes, hf, copd, patient_id=None):
        errors = []
        if not (18 <= age <= 90):    errors.append(f"age must be 18-90, got {age}")
        if not (1 <= los_days <= 60): errors.append(f"los_days must be 1-60, got {los_days}")
        if diabetes not in (0,1):    errors.append(f"diabetes must be 0 or 1")
        if hf not in (0,1):          errors.append(f"hf must be 0 or 1")
        if copd not in (0,1):        errors.append(f"copd must be 0 or 1")
        if errors: raise ValueError("; ".join(errors))
        self.age=age; self.los_days=los_days; self.diabetes=diabetes
        self.hf=hf; self.copd=copd; self.patient_id=patient_id

def predict_patient(patient, model_obj):
    X = [[patient.age,patient.los_days,patient.diabetes,patient.hf,patient.copd]]
    prob = float(model_obj.predict_proba(X)[0][1])
    tier = ("Low" if prob<0.10 else "Moderate" if prob<0.20
            else "High" if prob<0.35 else "Very High")
    return {"patient_id":patient.patient_id,"readmission_probability":round(prob,4),
            "risk_tier":tier,"model_version":"1.0.0"}

# Test valid and invalid patients
test_cases = [
    {"age":72,"los_days":8,"diabetes":1,"hf":1,"copd":0,"patient_id":"PT-00142","label":"High-risk HF patient"},
    {"age":35,"los_days":2,"diabetes":0,"hf":0,"copd":0,"patient_id":"PT-00289","label":"Low-risk young patient"},
    {"age":200,"los_days":5,"diabetes":0,"hf":0,"copd":0,"patient_id":"PT-bad","label":"Invalid age"},
    {"age":65,"los_days":10,"diabetes":2,"hf":0,"copd":0,"patient_id":"PT-bad2","label":"Invalid diabetes"},
]

print("FastAPI endpoint simulation (validation + prediction):")
print(f"{'Patient':35s} {'Status':8s} {'Prob':8s} {'Risk Tier'}")
print("-"*70)
for tc in test_cases:
    label = tc.pop("label")
    try:
        pt = PatientFeatures(**tc)
        result = predict_patient(pt, model)
        print(f"  {label:33s} OK       {result['readmission_probability']:.4f}  {result['risk_tier']}")
    except ValueError as e:
        print(f"  {label:33s} ERROR    {str(e)[:35]}")


## 5. API Test Suite

In [ ]:
# API testing patterns (simulating httpx/requests calls)
import json, time

def simulate_api_call(endpoint, payload, model_obj):
    t0 = time.time()
    try:
        if endpoint == "/health":
            return {"status":"healthy","timestamp":"2024-01-15T10:30:00"}
        elif endpoint == "/predict":
            from datetime import datetime
            pt_dict = payload
            if not all(k in pt_dict for k in ["age","los_days","diabetes","hf","copd"]):
                raise ValueError("Missing required fields")
            pt = PatientFeatures(**{k:pt_dict[k] for k in ["age","los_days","diabetes","hf","copd","patient_id"]
                                    if k in pt_dict})
            return predict_patient(pt, model_obj)
        elif endpoint == "/predict/batch":
            results = []
            for p in payload["patients"]:
                pt = PatientFeatures(**{k:p[k] for k in ["age","los_days","diabetes","hf","copd","patient_id"] if k in p})
                results.append(predict_patient(pt, model_obj))
            return {"predictions":results,"count":len(results)}
    except Exception as e:
        return {"error":str(e)}
    finally:
        elapsed = (time.time()-t0)*1000
        print(f"  [{endpoint}] {elapsed:.1f}ms")

print("API test suite simulation:")
print()
print("Test 1: Health check")
r = simulate_api_call("/health", {}, model)
print(f"  Response: {r}")

print("\nTest 2: Single prediction (high-risk)")
r = simulate_api_call("/predict",
    {"age":72,"los_days":12,"diabetes":1,"hf":1,"copd":0,"patient_id":"PT-00142"}, model)
print(f"  Response: {json.dumps(r,indent=4)}")

print("\nTest 3: Batch prediction (3 patients)")
r = simulate_api_call("/predict/batch",
    {"patients":[
        {"age":72,"los_days":12,"diabetes":1,"hf":1,"copd":0,"patient_id":"PT-001"},
        {"age":45,"los_days":3,"diabetes":0,"hf":0,"copd":0,"patient_id":"PT-002"},
        {"age":68,"los_days":7,"diabetes":1,"hf":0,"copd":1,"patient_id":"PT-003"},
    ]}, model)
for pred in r["predictions"]:
    print(f"  {pred['patient_id']}: prob={pred['readmission_probability']:.4f} | tier={pred['risk_tier']}")


## 6. Docker Deployment

In [ ]:
# Docker + deployment configuration
DOCKER_COMPOSE = (
    "version: '3.8'\n"
    "services:\n"
    "  api:\n"
    "    build: .\n"
    "    ports:\n"
    "      - '8000:8000'\n"
    "    environment:\n"
    "      - MODEL_PATH=/app/models/readmission_model.pkl\n"
    "      - LOG_LEVEL=INFO\n"
    "      - ALLOWED_ORIGINS=https://dashboard.healthsystem.org\n"
    "    volumes:\n"
    "      - ./models:/app/models:ro  # read-only model mount\n"
    "    healthcheck:\n"
    "      test: ['CMD','curl','-f','http://localhost:8000/health']\n"
    "      interval: 30s\n"
    "      timeout: 10s\n"
    "      retries: 3\n"
    "    restart: unless-stopped\n\n"
    "  nginx:\n"
    "    image: nginx:alpine\n"
    "    ports:\n"
    "      - '443:443'\n"
    "    volumes:\n"
    "      - ./nginx.conf:/etc/nginx/nginx.conf:ro\n"
    "      - /etc/ssl/certs:/etc/ssl/certs:ro\n"
    "    depends_on: [api]\n"
)

DOCKERFILE = (
    "FROM python:3.10-slim\n"
    "WORKDIR /app\n"
    "COPY requirements.txt .\n"
    "RUN pip install --no-cache-dir -r requirements.txt\n"
    "COPY src/ src/\n"
    "COPY main.py .\n"
    "RUN useradd -m appuser && chown -R appuser /app\n"
    "USER appuser\n"
    "EXPOSE 8000\n"
    "CMD ['uvicorn','main:app','--host','0.0.0.0','--port','8000','--workers','4']\n"
)

print("Dockerfile:")
print(DOCKERFILE)
print("\ndocker-compose.yml:")
print(DOCKER_COMPOSE)

Path("/tmp/mod08/Dockerfile").write_text(DOCKERFILE)
Path("/tmp/mod08/docker-compose.yml").write_text(DOCKER_COMPOSE)
print("Files written to /tmp/mod08/")


## 7. API Performance & Clinical Considerations

In [ ]:
# Latency benchmark (simulated)
import time
import numpy as np

BATCH_SIZES = [1, 10, 50, 100, 500]
latencies = []

test_patient = {"age":65,"los_days":7,"diabetes":1,"hf":0,"copd":0,"patient_id":"PT-test"}

for batch_size in BATCH_SIZES:
    patients = [test_patient.copy() for _ in range(batch_size)]
    t0 = time.time()
    for _ in range(20):  # 20 repeated calls
        X_batch = [[p["age"],p["los_days"],p["diabetes"],p["hf"],p["copd"]] for p in patients]
        model.predict_proba(X_batch)
    avg_ms = (time.time()-t0)/20*1000
    latencies.append(avg_ms)

fig, ax = plt.subplots(figsize=(9,4))
ax.plot(BATCH_SIZES, latencies, "o-", color="#4878CF", lw=2.5, ms=8)
ax.set_xlabel("Batch size (# patients)"); ax.set_ylabel("Latency (ms)")
ax.set_title("API Prediction Latency by Batch Size\n(GBM model, simulated endpoint)")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
for bs,lat in zip(BATCH_SIZES,latencies):
    ax.text(bs,lat+0.1,f"{lat:.1f}ms",ha="center",fontsize=8)
plt.tight_layout()
plt.savefig("/tmp/mod08/api_latency.png",bbox_inches="tight"); plt.show()

print("Clinical API requirements:")
print("  EHR integration    : < 500ms (synchronous calls within order workflow)")
print("  Batch discharge    : < 5s for 500 patients (overnight discharge planning)")
print("  Model loading      : one-time at startup, NOT per request (use Depends())")
print("  Rate limiting      : typically 100 req/min per client IP")
print("  PHI handling       : patient_id in request is optional; never log PHI in plaintext")


## 8. Knowledge Check
1. A clinician enters `age=-5` in the EHR interface. What HTTP status code should your API return and why?
2. Your batch endpoint receives 10,000 patients. How do you handle this without blocking other requests?
3. The API is integrated into the EHR. A code deploy breaks the `/predict` endpoint. What should your health check return?
4. A researcher wants to use the API for a study pulling 500,000 predictions. What rate limiting approach protects the production system?
5. Add an `/explain` endpoint that returns the top 3 SHAP feature contributions for a single patient prediction.


In [ ]:
# Exercise 5 — /explain endpoint with SHAP
try:
    import shap
    explainer = shap.TreeExplainer(model)
    SHAP_OK = True
except (ImportError, Exception):
    SHAP_OK = False

def explain_prediction(patient_dict, model_obj, features=FEATURES):
    X = pd.DataFrame([patient_dict[f] for f in features]).T
    X.columns = features
    prob = float(model_obj.predict_proba(X.values)[0][1])
    if SHAP_OK:
        exp = shap.TreeExplainer(model_obj)
        shap_vals = exp.shap_values(X.values)[0]
        contributions = sorted(zip(features, shap_vals), key=lambda x: abs(x[1]), reverse=True)[:3]
    else:
        coefs = model_obj.feature_importances_ if hasattr(model_obj,"feature_importances_") else np.ones(len(features))
        contributions = sorted(zip(features,coefs),key=lambda x:x[1],reverse=True)[:3]
    return {
        "patient_id"              : patient_dict.get("patient_id"),
        "readmission_probability" : round(prob,4),
        "risk_tier"               : "High" if prob>0.35 else "Moderate" if prob>0.20 else "Low",
        "top_3_drivers"           : [{"feature":f,"shap_value":round(float(v),4),"direction":"increases risk" if v>0 else "decreases risk"}
                                      for f,v in contributions],
    }

pt = {"age":72,"los_days":12,"diabetes":1,"hf":1,"copd":0,"patient_id":"PT-00142"}
result = explain_prediction(pt, model)
print("POST /explain response:")
print(json.dumps(result, indent=2))


---
**Next → NB-06: Testing, CI/CD & Code Quality**